# Hybrid DSVDD + Tag Prediction Training

This notebook trains a hybrid model that performs:
1. **Deep SVDD anomaly detection** - learns a hypersphere around normal samples
2. **Multi-label tag prediction** - predicts all 40 CelebA attributes

## Setup Instructions:
1. Upload your CelebA dataset to Google Drive at `MyDrive/CelebA/celeba/`
2. Make sure you have: `img_align_celeba.zip` and attribute files
3. Run all cells in order

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Clone Repository and Setup

In [ ]:
import os
import sys

# Change to content directory
%cd /content

# Clone the repository if it doesn't exist
if not os.path.exists('/content/tmlr-cxad'):
    !git clone https://github.com/Atticus-Wong/tmlr-cxad.git
    print("Repository cloned successfully!")
else:
    print("Repository already exists")

# Checkout the multitask branch
%cd /content/tmlr-cxad
!git fetch origin
!git checkout multitask
!git pull origin multitask

print("\nRepository setup complete!")

## Cell 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch torchvision scikit-learn matplotlib pillow

# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Cell 4: Setup Data Directories and Copy CelebA

In [ ]:
# Create directories
!mkdir -p /content/data/celeba
!mkdir -p /content/results

# Copy CelebA data from Google Drive
# Note: Make sure your CelebA data is at: MyDrive/CelebA/celeba/
drive_path = '/content/drive/MyDrive/CelebA/celeba'
local_path = '/content/data/celeba'

if os.path.exists(drive_path):
    !cp -r {drive_path}/* {local_path}/
    print(f"Copied data from {drive_path}")
else:
    print(f"WARNING: Drive path not found: {drive_path}")
    print("Please ensure CelebA data is uploaded to your Drive")

# Unzip images if needed
%cd /content/data/celeba
if os.path.exists('img_align_celeba.zip'):
    !unzip -q -o img_align_celeba.zip
    print("Unzipped img_align_celeba.zip")
else:
    print("img_align_celeba.zip not found - images may already be extracted")

%cd /content
print("\nData setup complete!")

## Cell 5: Add Repository to Python Path

In [ ]:
# Add the src directory to Python path
src_path = '/content/tmlr-cxad/CelebA/Deep-SVDD-PyTorch/src'
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Change to the src directory for proper imports
%cd /content/tmlr-cxad/CelebA/Deep-SVDD-PyTorch/src

print(f"Added {src_path} to Python path")
print(f"Current directory: {os.getcwd()}")

## Cell 6: Define Training Settings

Modify these settings as needed for your experiment.

In [ ]:
# Training configuration
settings = {
    "dataset_name": "celeba",
    "net_name": "celeba_hybrid_Net",  # Use hybrid network
    "xp_path": "/content/results",
    "data_path": "/content/data",
    "normal_class": 31,  # Smiling (0-54 for CelebA attributes)
    "objective": "one-class",  # or 'soft-boundary'
    "nu": 0.1,
    "seed": -1,  # -1 for random, or set to specific value for reproducibility
    "device": "cuda",
    "n_jobs_dataloader": 2,  # Number of workers for data loading
    "load_config": None,
    "load_model": None,
    
    # Training settings
    "pretrain": True,  # Pretrain with autoencoder first
    "train": True,
    "tag_loss_weight": 1.0,  # Weight for tag prediction loss
    
    # Optimizer settings
    "optimizer_name": "adam",
    "lr": 0.001,
    "n_epochs": 50,
    "lr_milestone": (),  # Learning rate decay milestones
    "batch_size": 128,
    "weight_decay": 1e-6,
    
    # Autoencoder pretraining settings
    "ae_optimizer_name": "adam",
    "ae_lr": 0.001,
    "ae_n_epochs": 100,
    "ae_lr_milestone": (),
    "ae_batch_size": 128,
    "ae_weight_decay": 1e-6,
}

# Display settings
print("Training Settings:")
print("=" * 50)
for key, value in settings.items():
    print(f"{key}: {value}")
print("=" * 50)

## Cell 7: Run Hybrid Training

This cell will start the training process. It includes:
- Autoencoder pretraining (if enabled)
- DSVDD + tag prediction training
- Evaluation on test set
- Visualization of results

In [ ]:
# Run the hybrid training
# This will execute the hybrid_train.py script with your settings

exec(open('hybrid_train.py').read())

## Cell 8: Save Results to Drive (Optional)

Copy results back to Google Drive for persistent storage.

In [ ]:
# Save results to Drive
results_drive_path = '/content/drive/MyDrive/DSVDD_Results'
!mkdir -p {results_drive_path}

# Copy results
!cp -r /content/results/* {results_drive_path}/

print(f"Results saved to: {results_drive_path}")
print("\nFiles saved:")
!ls -lh {results_drive_path}/

## Cell 9: View Results

Display the generated visualizations and results.

In [ ]:
from IPython.display import Image, display

# Display normal examples
print("Most Normal Examples:")
display(Image('/content/results/normals.png'))

print("\nMost Anomalous Examples:")
display(Image('/content/results/outliers.png'))

print("\nAttribute Frequency (Outliers):")
display(Image('/content/results/frequency_outliers.png'))

print("\nAttribute Frequency (Normals):")
display(Image('/content/results/frequency_normals.png'))

## Cell 10: Check Training Logs

In [ ]:
# Display training log
with open('/content/results/log.txt', 'r') as f:
    log_content = f.read()
    print(log_content[-2000:])  # Print last 2000 characters

## Troubleshooting

### Common Issues:

1. **Out of Memory Error**
   - Reduce `batch_size` in settings (try 64 or 32)
   - Reduce `ae_n_epochs` or `n_epochs`

2. **Drive Not Mounted**
   - Re-run Cell 1 to mount Drive
   - Ensure you authorized access

3. **CelebA Data Not Found**
   - Check that data is at `MyDrive/CelebA/celeba/`
   - Verify `img_align_celeba.zip` exists

4. **CUDA Out of Memory**
   - Restart runtime: Runtime > Restart runtime
   - Reduce batch size
   - Use `device: 'cpu'` in settings (slower but works)

### Normal Class Indices for CelebA:
- 31: Smiling
- 20: Male
- 15: Eyeglasses
- 35: Wearing_Hat
- See `list_attr_celeba.txt` for all 40 attributes